# LeetCode 4007 — Widest Possible Fence bằng Python

- Link: https://leetcode.com/problems/widest-possible-fence/
- Độ khó: **Hard**
- Chủ đề: Đếm tần suất (Counting), Hash Table, Liệt kê cặp (Enumeration)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumWidth(self, planks: list[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bạn có nhiều tấm ván, mỗi tấm rộng 1 đơn vị và cao `planks[i]`. Bạn muốn dựng một hàng rào gồm các tấm có **cùng chiều cao**. Mỗi tấm của hàng rào là một ván nguyên **hoặc** ghép **đúng hai** ván khác nhau (cộng chiều cao). Mỗi ván gốc dùng tối đa một lần. Hỏi hàng rào rộng nhất bao nhiêu.

Bài được xếp Hard nhưng lời giải cuối cùng rất ngắn. Cái khó nằm ở chỗ **lập luận**: vì sao các "nguồn" tạo ra chiều cao `h` không tranh nhau ván, nên cộng thẳng được.

Sau notebook này bạn sẽ:

1. Dùng `Counter` và `defaultdict` để đếm và cộng dồn.
2. Hiểu ba nguồn tạo ra tấm cao `h`: dùng nguyên, ghép hai tấm cùng cao `h/2`, ghép hai tấm cao `x` và `h-x`.
3. Chứng minh ba nguồn (và các cặp `(x, h-x)` khác nhau) **không dùng chung ván**, nên số tấm là tổng.
4. Viết vét cạn bitmask làm trọng tài, cách theo từng chiều cao mục tiêu `O(m³)`, và cách cộng dồn `O(m²)`.
5. Đối chiếu các cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `planks`. Mỗi tấm của hàng rào bạn dựng phải thuộc một trong hai loại:

- một ván gốc dùng nguyên chiều cao của nó;
- ghép **đúng hai ván gốc khác nhau** (khác chỉ số) thành một tấm có chiều cao bằng tổng.

Toàn bộ tấm trong hàng rào phải **cùng chiều cao**. Mỗi ván gốc được dùng tối đa một lần, không cần dùng hết. Trả về **độ rộng lớn nhất** (số tấm nhiều nhất).

**Ví dụ 1:** `planks = [1,3,2,5,7,5,4,2,1]` → `4`. Dựng 4 tấm cao 5: hai ván `5` nguyên, `1 + 4`, `3 + 2`.

**Ví dụ 2:** `planks = [2,3,7]` → `1`. Không thể tạo hai tấm cùng cao, nên lấy bất kỳ một ván nguyên.

**Ràng buộc**

- `1 <= planks.length <= 1000`
- `1 <= planks[i] <= 10^9`

Câu hỏi nhanh: đáp án luôn `>= 1`, vì sao? (Vì bạn luôn chọn được một ván nguyên.)

In [1]:
from typing import List

EXAMPLES = [
    ([1, 3, 2, 5, 7, 5, 4, 2, 1], 4),
    ([2, 3, 7], 1),
]
for planks, expected in EXAMPLES:
    print(f"planks={planks} -> kỳ vọng {expected}")

planks=[1, 3, 2, 5, 7, 5, 4, 2, 1] -> kỳ vọng 4
planks=[2, 3, 7] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. `Counter` — đếm số lần xuất hiện

`collections.Counter` nhận một danh sách và trả về dict `{giá trị: số lần}`. Truy cập giá trị chưa có cho ra `0` (không gây lỗi).

In [2]:
from collections import Counter, defaultdict

cnt = Counter([1, 3, 2, 5, 7, 5, 4, 2, 1])
print(cnt)                 # Counter({1: 2, 2: 2, 5: 2, 3: 1, 7: 1, 4: 1})
print(cnt[5], cnt[100])    # 2 0  (không có thì là 0)
print(list(cnt.items()))   # cặp (giá trị, số lần)
print(sorted(cnt))         # các giá trị phân biệt, đã sắp xếp

Counter({1: 2, 2: 2, 5: 2, 3: 1, 7: 1, 4: 1})
2 0
[(1, 2), (3, 1), (2, 2), (5, 2), (7, 1), (4, 1)]
[1, 2, 3, 4, 5, 7]


### 3.2. `defaultdict(int)` — cộng dồn không cần kiểm tra

`defaultdict(int)` tạo dict mà khoá chưa có sẽ tự có giá trị `0`. Nhờ vậy ta viết `t[h] += 5` ngay mà không cần `if h in t`.

In [3]:
t = defaultdict(int)
for h, add in [(5, 2), (4, 1), (5, 1), (5, 1)]:
    t[h] += add
print(dict(t))             # {5: 4, 4: 1}
print(max(t.values()))     # 4

{5: 4, 4: 1}
4


### 3.3. Phép chia nguyên `//` và `min` của hai số lần

- `v // 2` = số **cặp** có thể ghép từ `v` tấm cùng cao (mỗi cặp dùng 2 tấm, dư 1 nếu `v` lẻ).
- Ghép tấm cao `x` với tấm cao `y` (khác nhau) tạo được tối đa `min(cnt[x], cnt[y])` tấm, vì mỗi tấm mới dùng một `x` và một `y`.

In [4]:
print([(v, v // 2) for v in range(0, 6)])   # số cặp từ v tấm cùng cao
cx, cy = 3, 5
print("ghép x(3 tấm) với y(5 tấm) được tối đa", min(cx, cy), "tấm")

[(0, 0), (1, 0), (2, 1), (3, 1), (4, 2), (5, 2)]
ghép x(3 tấm) với y(5 tấm) được tối đa 3 tấm


## 4. Ý tưởng nền tảng / trực giác

Hãy cố định chiều cao mục tiêu `h` và hỏi: **với `h` này, dựng được bao nhiêu tấm?** Đáp án cuối cùng là `max` theo mọi `h`.

Ví dụ 1 với `h = 5`. Có ván cao: `1,1,2,2,3,4,5,5,7`.

- Ván nguyên cao 5: có hai ván → 2 tấm.
- Ghép `1 + 4`: có hai ván cao 1, một ván cao 4 → `min(2, 1) = 1` tấm.
- Ghép `2 + 3`: hai ván cao 2, một ván cao 3 → `min(2, 1) = 1` tấm.
- `h` lẻ nên không có "hai tấm cùng cao `h/2`".

Tổng: `2 + 1 + 1 = 4`. Đúng với đáp án.

Câu hỏi tự nhiên: ta có cộng được `2 + 1 + 1` không? Nếu ván cao 1 dùng ở hai nơi thì sẽ bị tính hai lần. Ta sẽ chứng minh điều đó **không xảy ra** ở mục 6.

In [5]:
cnt = Counter([1, 3, 2, 5, 7, 5, 4, 2, 1])
h = 5
direct = cnt[h]
pairs = sum(min(cnt[x], cnt[h - x]) for x in cnt if x < h - x)      # x < y để mỗi cặp (x, y) chỉ tính một lần
half = cnt[h // 2] // 2 if h % 2 == 0 else 0
print("trực tiếp:", direct, "| ghép khác chiều:", pairs, "| ghép hai tấm cùng cao h/2:", half)
print("tổng cho h =", h, ":", direct + pairs + half)

trực tiếp: 2 | ghép khác chiều: 2 | ghép hai tấm cùng cao h/2: 0
tổng cho h = 5 : 4


## 5. Cách 1 — Brute Force theo định nghĩa (bitmask)

**Ý tưởng.** Chiều cao mục tiêu `h` chỉ có thể là một ván nguyên hoặc một tổng hai ván: nên có tối đa `n + C(n, 2)` ứng viên. Với mỗi `h`, ta đệ quy trên **từng ván theo chỉ số**: ván `i` hoặc bị bỏ, hoặc dùng nguyên (nếu `= h`), hoặc ghép với một ván `j > i` chưa dùng (nếu tổng `= h`). Trả về số tấm nhiều nhất.

Cách này làm đúng theo đề từng bước nên rất đáng tin, nhưng chỉ chạy nổi với `n ≤ 10`.

In [6]:
from functools import lru_cache

def brute_force(planks):
    n = len(planks)
    candidates = set(planks)
    for i in range(n):
        for j in range(i + 1, n):
            candidates.add(planks[i] + planks[j])

    best = 0
    for h in candidates:
        @lru_cache(None)
        def go(mask):
            # mask: các ván đã xử lý (bit i = 1). Tìm số tấm nhiều nhất cho các ván còn lại
            if mask == (1 << n) - 1:
                return 0
            i = 0
            while mask >> i & 1:
                i += 1
            res = go(mask | 1 << i)                    # bỏ ván i
            if planks[i] == h:                         # dùng nguyên
                res = max(res, 1 + go(mask | 1 << i))
            for j in range(i + 1, n):                  # ghép i với j
                if not mask >> j & 1 and planks[i] + planks[j] == h:
                    res = max(res, 1 + go(mask | 1 << i | 1 << j))
            return res
        best = max(best, go(0))
    return best

for planks, expected in EXAMPLES:
    print(planks, "->", brute_force(planks), "(kỳ vọng", expected, ")")

[1, 3, 2, 5, 7, 5, 4, 2, 1] -> 4 (kỳ vọng 4 )
[2, 3, 7] -> 1 (kỳ vọng 1 )


**Độ phức tạp.** Số ứng viên `O(n²)`, mỗi ứng viên có tối đa `2^n` mặt nạ, mỗi mặt nạ `O(n)` lựa chọn → cỡ `O(n² · 2^n · n)`. Với `n = 1000` là bất khả thi. **Nút thắt:** ta đang xét từng ván riêng lẻ, trong khi các ván **cùng chiều cao hoàn toàn giống nhau**. Chỉ cần biết mỗi chiều cao có bao nhiêu ván.

## 6. Bước đột phá — ba nguồn không tranh nhau ván

Gom ván theo chiều cao: `cnt[x]` = số ván cao `x`. Với mục tiêu `h`, các cách tạo tấm cao `h`:

1. **Nguyên**: ván cao `h` → `cnt[h]` tấm.
2. **Hai tấm cùng cao `h/2`** (khi `h` chẵn): mỗi tấm dùng 2 ván cao `h/2` → `cnt[h/2] // 2` tấm.
3. **Hai chiều cao khác nhau `x < y`, `x + y = h`**: mỗi tấm dùng một ván `x` và một `y` → `min(cnt[x], cnt[y])` tấm.

**Tại sao cộng thẳng được?** Xét xem mỗi nguồn dùng những chiều cao nào:

- Nguồn 1 dùng chiều cao `h`.
- Nguồn 2 dùng chiều cao `h/2`.
- Nguồn 3 với cặp `(x, h-x)` dùng chiều cao `x` và `h-x`.

Vì mọi chiều cao đều **dương**, `x < h` và `h-x < h`, nên nguồn 1 (chiều cao đúng bằng `h`) khác mọi chiều cao trong nguồn 2 và 3. Giữa các cặp `(x, h-x)` khác nhau (`x` khác nhau, cùng quy ước `x < h-x`), tập chiều cao `{x, h-x}` **rời nhau** vì `x` xác định `h-x`. Còn nguồn 2 dùng `h/2`, là trường hợp `x = h-x`, cũng rời nhau với các cặp `x < h-x`.

Vậy các nguồn dùng **các nhóm ván rời nhau**; tối đa hoá riêng từng nhóm rồi cộng lại là tối ưu (đó là bước then chốt, chứng minh chi tiết ở mục 10).

Hãy kiểm chứng bằng số liệu: công thức cộng theo từng `h` có khớp brute force?

In [7]:
def width_for_target(cnt, h):
    total = cnt.get(h, 0)
    if h % 2 == 0:
        total += cnt.get(h // 2, 0) // 2
    for x in cnt:
        y = h - x
        if x < y and y in cnt:
            total += min(cnt[x], cnt[y])
    return total

cnt = Counter([1, 3, 2, 5, 7, 5, 4, 2, 1])
print({h: width_for_target(cnt, h) for h in [5, 3, 4, 2, 7, 6]})
print("max theo mọi h có thể:", max(width_for_target(cnt, h) for h in set(cnt) | {x + y for x in cnt for y in cnt if x != y or cnt[x] >= 2}))

{5: 4, 3: 3, 4: 3, 2: 3, 7: 4, 6: 3}
max theo mọi h có thể: 4


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt từng chiều cao mục tiêu `h`

**Giải thích từng đoạn.**

1. `cnt = Counter(planks)`: đếm.
2. Tập ứng viên `H`: mọi chiều cao có sẵn và mọi tổng `x + y` của hai chiều cao khác nhau, cùng `2x` khi `cnt[x] >= 2`.
3. Với mỗi `h` trong `H`, tính `width_for_target` ở trên (`O(m)` vì lặp qua các chiều cao phân biệt `x`).
4. Đáp án = `max`.

Nó đúng, dễ hiểu, nhưng số ứng viên là `O(m²)` và mỗi ứng viên tốn `O(m)` nên `O(m³)`.

In [8]:
def by_target(planks):
    cnt = Counter(planks)
    values = sorted(cnt)
    candidates = set(values)
    for i, x in enumerate(values):
        if cnt[x] >= 2:
            candidates.add(2 * x)
        for y in values[i + 1:]:
            candidates.add(x + y)
    return max(width_for_target(cnt, h) for h in candidates)

for planks, expected in EXAMPLES:
    assert by_target(planks) == expected
print("Cách 2 khớp ví dụ")

Cách 2 khớp ví dụ


**Độ phức tạp.** `O(m³)` thời gian (với `m` là số chiều cao phân biệt, `m ≤ n ≤ 1000` → có thể tới `10^9`, quá chậm), `O(m²)` bộ nhớ cho tập ứng viên.

### 7.2. Cách 3 — Cộng dồn theo cặp chiều cao (đáp án nộp)

**Ý tưởng.** Thay vì với mỗi `h` đi hỏi "những ai đóng góp?", ta đảo ngược: **mỗi đóng góp tự báo cáo cho `h` của nó** vào bảng `t`.

- Mỗi chiều cao `x`: `t[x] += cnt[x]` (dùng nguyên) và `t[2x] += cnt[x] // 2` (ghép hai tấm cùng cao).
- Mỗi cặp `x < y`: `t[x + y] += min(cnt[x], cnt[y])`.

Đáp án = `max(t.values())`. Mỗi cặp chỉ được thăm **một lần**, tổng `O(m²)`.

**Giải thích code.**

1. `for x, v1 in cnt.items()` — vòng ngoài lấy chiều cao `x` và số tấm `v1`.
2. Hai dòng cộng dồn đầu: nguyên và cặp cùng cao.
3. `for y, v2 in cnt.items(): if y > x:` — vòng trong chỉ lấy `y > x` để mỗi cặp xét một lần.
4. `t[x + y] += min(v1, v2)`.

In [9]:
def accumulate(planks):
    cnt = Counter(planks)
    t = defaultdict(int)
    for x, v1 in cnt.items():
        t[x] += v1
        t[2 * x] += v1 // 2
        for y, v2 in cnt.items():
            if y > x:
                t[x + y] += min(v1, v2)
    return max(t.values())

for planks, expected in EXAMPLES:
    assert accumulate(planks) == expected
print("Cách 3 khớp ví dụ")

Cách 3 khớp ví dụ


**Độ phức tạp.** `O(n + m²)` thời gian (đếm `O(n)`, hai vòng lồng `O(m²)`), `O(m²)` bộ nhớ nếu mọi tổng khác nhau (bảng `t` có thể chứa tới `m²/2` khoá). Với `m ≤ 1000` chỉ ~500 nghìn cặp nên rất nhanh.

Chú ý: đáp án của doocs ghi bộ nhớ `O(m)` nhưng bảng `t` thực tế có thể lớn tới `O(m²)` khoá; không ảnh hưởng gì với giới hạn đề, nhưng ta ghi đúng ở đây.

### 7.3. Cách 4 — Cộng dồn trên danh sách đã sắp xếp (tránh dict lồng)

**Ý tưởng.** Sắp xếp các chiều cao phân biệt rồi dùng hai chỉ số `i < j`; tương đương cách 3 nhưng cho thứ tự duyệt xác định và dễ debug. Không nhanh hơn về bậc, nhưng giúp ta trả thêm được **chiều cao tối ưu** (mục biến thể).

In [10]:
def accumulate_sorted(planks):
    cnt = Counter(planks)
    vals = sorted(cnt)
    t = defaultdict(int)
    for i, x in enumerate(vals):
        t[x] += cnt[x]
        t[2 * x] += cnt[x] // 2
        for j in range(i + 1, len(vals)):
            y = vals[j]
            t[x + y] += min(cnt[x], cnt[y])
    return max(t.values())

for planks, expected in EXAMPLES:
    assert accumulate_sorted(planks) == expected
print("Cách 4 khớp ví dụ")

Cách 4 khớp ví dụ


## 8. TRACE từng bước

Ta in bảng `t` sau mỗi chiều cao `x` được xử lý, với ví dụ 1. Cột "đóng góp" cho biết `x` đã thêm vào những `h` nào.

| bước | ý nghĩa |
|---|---|
| nguyên | `t[x] += cnt[x]` |
| cặp cùng cao | `t[2x] += cnt[x] // 2` |
| cặp khác cao | `t[x+y] += min(cnt[x], cnt[y])`, `y > x` |

In [11]:
def trace_accumulate(planks, watch=None):
    cnt = Counter(planks)
    vals = sorted(cnt)
    print("cnt =", dict((v, cnt[v]) for v in vals))
    t = defaultdict(int)
    for i, x in enumerate(vals):
        log = []
        t[x] += cnt[x];        log.append(f"t[{x}] += {cnt[x]}")
        if cnt[x] // 2:
            t[2 * x] += cnt[x] // 2; log.append(f"t[{2 * x}] += {cnt[x] // 2}")
        for y in vals[i + 1:]:
            m = min(cnt[x], cnt[y])
            t[x + y] += m;     log.append(f"t[{x + y}] += min({cnt[x]},{cnt[y]})={m}")
        print(f"\nx={x} (có {cnt[x]} ván):  " + ", ".join(log))
        if watch is not None:
            print(f"   -> t[{watch}] hiện là {t[watch]}")
    best = max(t.values())
    print("\nbảng t cuối:", dict(sorted(t.items())))
    print("các h đạt max:", [h for h, v in sorted(t.items()) if v == best], "-> đáp án", best)
    return best

trace_accumulate([1, 3, 2, 5, 7, 5, 4, 2, 1], watch=5)

cnt = {1: 2, 2: 2, 3: 1, 4: 1, 5: 2, 7: 1}

x=1 (có 2 ván):  t[1] += 2, t[2] += 1, t[3] += min(2,2)=2, t[4] += min(2,1)=1, t[5] += min(2,1)=1, t[6] += min(2,2)=2, t[8] += min(2,1)=1
   -> t[5] hiện là 1

x=2 (có 2 ván):  t[2] += 2, t[4] += 1, t[5] += min(2,1)=1, t[6] += min(2,1)=1, t[7] += min(2,2)=2, t[9] += min(2,1)=1
   -> t[5] hiện là 2

x=3 (có 1 ván):  t[3] += 1, t[7] += min(1,1)=1, t[8] += min(1,2)=1, t[10] += min(1,1)=1
   -> t[5] hiện là 2

x=4 (có 1 ván):  t[4] += 1, t[9] += min(1,2)=1, t[11] += min(1,1)=1
   -> t[5] hiện là 2

x=5 (có 2 ván):  t[5] += 2, t[10] += 1, t[12] += min(2,1)=1
   -> t[5] hiện là 4

x=7 (có 1 ván):  t[7] += 1
   -> t[5] hiện là 4

bảng t cuối: {1: 2, 2: 3, 3: 3, 4: 3, 5: 4, 6: 3, 7: 4, 8: 2, 9: 2, 10: 2, 11: 1, 12: 1}
các h đạt max: [5, 7] -> đáp án 4


4

Quan sát `t[5]`: đóng góp lần lượt từ `x=1` (cặp `1+4` → 1), `x=2` (cặp `2+3` → 1), và `x=5` (nguyên → 2). Tổng 4, đúng với đề. Trace thứ hai với ví dụ 2 cho thấy đáp án `1`:

In [12]:
trace_accumulate([2, 3, 7])

cnt = {2: 1, 3: 1, 7: 1}

x=2 (có 1 ván):  t[2] += 1, t[5] += min(1,1)=1, t[9] += min(1,1)=1

x=3 (có 1 ván):  t[3] += 1, t[10] += min(1,1)=1

x=7 (có 1 ván):  t[7] += 1

bảng t cuối: {2: 1, 3: 1, 5: 1, 7: 1, 9: 1, 10: 1}
các h đạt max: [2, 3, 5, 7, 9, 10] -> đáp án 1


1

## 9. Code nộp LeetCode

In [13]:
from typing import List
from collections import Counter, defaultdict

class Solution:
    def maximumWidth(self, planks: List[int]) -> int:
        cnt = Counter(planks)
        t = defaultdict(int)
        for x, v1 in cnt.items():
            t[x] += v1                    # dùng nguyên ván cao x
            t[2 * x] += v1 // 2           # ghép hai ván cùng cao x
            for y, v2 in cnt.items():
                if y > x:                 # mỗi cặp chiều cao khác nhau chỉ xét một lần
                    t[x + y] += min(v1, v2)
        return max(t.values())

for planks, expected in EXAMPLES:
    assert Solution().maximumWidth(planks) == expected
print("Code nộp khớp ví dụ")

Code nộp khớp ví dụ


## 10. Vì sao thuật toán đúng

**Mô hình.** Một hàng rào hợp lệ chiều cao `h` là: một tập tấm, mỗi tấm là (a) một ván cao `h`, hoặc (b) một cặp ván `(p, q)` với `p + q = h`. Các ván trong các tấm khác nhau phải khác nhau.

**Bước 1: phân rã theo chiều cao.** Một ván cao `x` chỉ có thể xuất hiện trong: tấm kiểu (a) nếu `x = h`; hoặc tấm kiểu (b) với bạn đời cao `h - x`. Vì `x ≥ 1` và `h - x ≥ 1` nên `x < h`: ván kiểu (a) **không** thể dùng ở kiểu (b). Còn ván cao `x < h` chỉ có đúng một "bạn đời" chiều cao `h - x`.

**Bước 2: các nhóm rời nhau.** Với `h` cố định, chia các chiều cao thành các nhóm: `{h}`, và `{x, h - x}` với `x < h - x`, và (nếu `h` chẵn) `{h/2}`. Hai nhóm bất kỳ không có chiều cao chung, vì `x` quyết định `h - x` và ngược lại. Mọi tấm hợp lệ chỉ lấy ván trong **đúng một** nhóm.

**Bước 3: tối ưu từng nhóm.**

- Nhóm `{h}`: tất cả `cnt[h]` ván dùng nguyên.
- Nhóm `{h/2}`: mỗi tấm dùng hai ván, tối đa `cnt[h/2] // 2`.
- Nhóm `{x, y}` với `x ≠ y`: mỗi tấm dùng đúng một `x` và một `y`, tối đa `min(cnt[x], cnt[y])` (ghép bất kỳ `x` với `y`).

**Bước 4: kết luận.** Độ rộng tối ưu cho `h` là tổng ba loại. Đáp án = `max` theo `h`. Với mọi `h` mà bảng `t` có khoá, ta đã cộng đúng các đóng góp; với `h` không phải ván nào và không là tổng nào thì không có tấm nào. Ngoài ra `t[x]` luôn `≥ cnt[x] ≥ 1` nên `max(t.values())` luôn tồn tại và `≥ 1`.

**Bước 5: vì sao mỗi cặp chỉ cộng một lần.** Vòng trong chỉ nhận `y > x`, nên cặp `{x, y}` chỉ góp một lần vào `t[x + y]`. Còn `t[2x]` nhận `cnt[x] // 2` khi ghép cùng cao; không bị lặp với `y > x`.

## 11. Phân tích Time/Space

Gọi `n = len(planks)`, `m` = số chiều cao phân biệt (`m ≤ n`).

| Cách | Time | Space |
|---|---|---|
| Brute force bitmask | `O(n² · 2^n · n)` | `O(2^n)` |
| Duyệt từng `h` | `O(m³)` | `O(m²)` |
| Cộng dồn (nộp) | `O(n + m²)` | `O(m²)` |
| Cộng dồn trên danh sách sắp xếp | `O(n log n + m²)` | `O(m²)` |

Trường hợp xấu nhất: `n = 1000` ván đều khác nhau → `m = 1000`, ~500 nghìn cặp. Ta đo thời gian:

In [14]:
import time, random

rng = random.Random(0)
planks = rng.sample(range(1, 10**9), 1000)
t0 = time.time()
ans = Solution().maximumWidth(planks)
print("n=1000 phân biệt hoàn toàn: đáp án", ans, "thời gian", round(time.time() - t0, 3), "giây")

planks = [rng.randint(1, 50) for _ in range(1000)]
t0 = time.time()
print("n=1000, giá trị 1..50: đáp án", Solution().maximumWidth(planks), "thời gian", round(time.time() - t0, 4), "giây")

n=1000 phân biệt hoàn toàn: đáp án 2 thời gian 0.223 giây
n=1000, giá trị 1..50: đáp án 463 thời gian 0.0002 giây


## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ (cho các cách làm sai) và đối chiếu ngẫu nhiên với brute force.

In [15]:
S = Solution()
# Ví dụ đề
for planks, expected in EXAMPLES:
    assert S.maximumWidth(planks) == expected

# Edge case
assert S.maximumWidth([7]) == 1                  # một ván
assert S.maximumWidth([5, 5]) == 2               # hai ván cùng cao, dùng nguyên
assert S.maximumWidth([1, 1]) == 2               # 1 và 1: hoặc 2 tấm cao 1, hoặc 1 tấm cao 2 -> chọn 2
assert S.maximumWidth([1, 1, 1, 1]) == 4         # cả 4 tấm cao 1 (tốt hơn 2 tấm cao 2)
assert S.maximumWidth([2, 2, 2, 2, 4, 4]) == 4   # h=4: 2 ván cao 4 + 2 cặp (2+2) = 4
assert S.maximumWidth([1, 2, 3, 4, 5, 6]) == 3   # h=7: 1+6, 2+5, 3+4
assert S.maximumWidth([10**9, 10**9]) == 2       # giá trị lớn
print("OK các edge case")

OK các edge case


**Một edge case tinh tế.** `planks = [1, 2, 3]`, `h = 3`: ván nguyên cao 3 → 1 tấm; ghép `1 + 2 = 3` → 1 tấm. Tổng **2 tấm cao 3**, nên đáp án là `2` (không phải `1`). Ngược lại `[2, 3, 7]` chỉ được `1`. Ta kiểm tra trực tiếp với brute force:

In [16]:
print(Solution().maximumWidth([1, 2, 3]), brute_force([1, 2, 3]))   # cả hai phải cho 2

2 2


In [17]:
assert S.maximumWidth([1, 2, 3]) == 2
assert S.maximumWidth([2, 3, 7]) == 1
assert S.maximumWidth([1, 2, 3, 3]) == 3         # h=3: hai ván 3 + cặp (1+2)
print("Edge case đã sửa: OK")

Edge case đã sửa: OK


**Phản ví dụ cho các cách nghĩ sai.**

1. Chỉ đếm `max(cnt.values())` (không ghép) → sai ở ví dụ 1 (`max cnt = 2` nhưng đáp án 4).
2. Quên chia `//2` khi ghép cùng cao: `[1,1,1]`, `h=2` chỉ được 1 tấm.
3. Cho `y >= x` ở vòng trong mà không xử lý `x == y` riêng: cặp `(x, x)` sẽ bị tính `min(v, v) = v` thay vì `v // 2`.

In [18]:
def wrong_no_pair(planks):
    return max(Counter(planks).values())

def wrong_same_pair(planks):
    cnt = Counter(planks)
    t = defaultdict(int)
    for x, v1 in cnt.items():
        t[x] += v1
        for y, v2 in cnt.items():
            if y >= x:                    # SAI: x == y bị tính min(v, v) = v
                t[x + y] += min(v1, v2)
    return max(t.values())

print("không ghép      :", wrong_no_pair(EXAMPLES[0][0]), " đúng:", Solution().maximumWidth(EXAMPLES[0][0]))
print("ghép cùng cao sai:", wrong_same_pair([1, 1, 1]), " đúng:", Solution().maximumWidth([1, 1, 1]))

không ghép      : 2  đúng: 4
ghép cùng cao sai: 3  đúng: 3


In [19]:
# Đối chiếu ngẫu nhiên có seed: 4 cách + class nộp với brute force
rng = random.Random(2026)
for it in range(1200):
    n = rng.randint(1, 9)
    hi = rng.choice([3, 5, 8, 20])
    planks = [rng.randint(1, hi) for _ in range(n)]
    ref = brute_force(planks)
    assert by_target(planks) == ref, planks
    assert accumulate(planks) == ref, planks
    assert accumulate_sorted(planks) == ref, planks
    assert S.maximumWidth(planks) == ref, planks
print("1200 input ngẫu nhiên nhỏ: các cách khớp brute force")

# Input lớn: so sánh cách 2 (O(m^3), dùng m nhỏ) với cách nộp
for it in range(40):
    n = rng.randint(50, 120)
    planks = [rng.randint(1, 60) for _ in range(n)]
    assert by_target(planks) == S.maximumWidth(planks)
print("40 input vừa: cách 2 khớp cách nộp")

1200 input ngẫu nhiên nhỏ: các cách khớp brute force
40 input vừa: cách 2 khớp cách nộp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm chiều cao xuất hiện nhiều nhất

Nghĩ rằng "hàng rào cùng cao" nghĩa là các ván có sẵn cùng cao, quên rằng có thể ghép.

In [20]:
planks = [1, 3, 2, 5, 7, 5, 4, 2, 1]
print("sai :", max(Counter(planks).values()))    # 2
print("đúng:", Solution().maximumWidth(planks))   # 4

sai : 2
đúng: 4


**Vì sao sai:** đề cho phép ghép hai ván khác nhau. **Cách đúng:** xét từng chiều cao mục tiêu `h` và cộng ba nguồn.

### Lỗi 2 — Cặp cùng chiều cao tính `min(v, v) = v`

Với `x == y` mỗi tấm ghép dùng **hai** ván cùng cao, nên chỉ được `v // 2`.

In [21]:
cnt = Counter([4, 4, 4, 4, 4])
print("sai : t[8] =", min(cnt[4], cnt[4]))      # 5
print("đúng: t[8] =", cnt[4] // 2)               # 2
print("đáp án thật (5 ván cao 4):", Solution().maximumWidth([4, 4, 4, 4, 4]))   # 5 (dùng nguyên tốt hơn)

sai : t[8] = 5
đúng: t[8] = 2
đáp án thật (5 ván cao 4): 5


**Cách đúng:** tách riêng `t[2x] += v1 // 2`, và vòng trong chỉ nhận `y > x`.

### Lỗi 3 — Duyệt cặp hai chiều: mỗi cặp bị cộng hai lần

Nếu vòng trong là `for y in cnt: if y != x:`, cặp `(x, y)` được cộng ở cả lượt `x` lẫn lượt `y` → `t[x+y]` gấp đôi.

In [22]:
def wrong_double(planks):
    cnt = Counter(planks)
    t = defaultdict(int)
    for x, v1 in cnt.items():
        t[x] += v1
        t[2 * x] += v1 // 2
        for y, v2 in cnt.items():
            if y != x:                    # SAI: cả (x, y) lẫn (y, x)
                t[x + y] += min(v1, v2)
    return max(t.values())

print("sai :", wrong_double([1, 2, 3, 3]))               # 4: đếm đôi
print("đúng:", Solution().maximumWidth([1, 2, 3, 3]))    # 3

sai : 4
đúng: 3


**Cách đúng:** dùng điều kiện `y > x` (hoặc duyệt `i < j` trên danh sách sắp xếp).

### Lỗi 4 — Quên rằng mỗi ván gốc dùng một lần khi trộn nhiều nguồn

Nếu ta gộp các cặp khác nhau của *cùng một chiều cao* (ví dụ `1+4` và `1+4` ở hai `h` khác nhau) thì phải nhớ ta chỉ chọn **một** `h` cho cả hàng rào: đáp án là `max` theo `h`, không phải tổng tất cả `t[h]`.

In [23]:
cnt = Counter([1, 3, 2, 5, 7, 5, 4, 2, 1])
t = defaultdict(int)
for x, v1 in cnt.items():
    t[x] += v1
    t[2 * x] += v1 // 2
    for y, v2 in cnt.items():
        if y > x:
            t[x + y] += min(v1, v2)
print("sai (cộng mọi h):", sum(t.values()))
print("đúng (max theo h):", max(t.values()))

sai (cộng mọi h): 30
đúng (max theo h): 4


## 14. Mẹo & nhận dạng pattern

- **"Các phần tử đồng nhất"**: nếu nhiều phần tử giống hệt nhau (cùng chiều cao), hãy đếm tần suất rồi làm việc trên các giá trị phân biệt (`m ≤ n`).
- **"Ghép hai phần tử có tổng `h`"**: liên hệ với Two Sum; ở đây cần *số lượng* cặp, không chỉ có/không.
- **Chứng minh "cộng thẳng được"** bằng cách chỉ ra các nhóm ván **rời nhau** — mẫu lập luận quen thuộc ở các bài ghép cặp.
- **Đảo vai**: thay vì với mỗi `h` đi tìm đóng góp, để mỗi đóng góp tự đẩy vào `t[h]`. Đây là kỹ thuật "scatter" so với "gather".
- **`n ≤ 1000` và `O(m²)`** là tín hiệu rõ ràng rằng đáp án mong đợi là hai vòng lồng nhau.
- Luôn tự hỏi đáp án nhỏ nhất có thể (ở đây `1`) và viết sẵn test cho nó.

## 15. Biến thể

1. **Trả cả chiều cao tối ưu** (nhỏ nhất khi hoà) — chỉ cần `argmax` trên `t`.
2. **Chỉ được ghép các ván khác chiều cao** (cấm cặp `(x, x)`): bỏ dòng `t[2x] += ...`.
3. **Cho ghép tối đa `k` ván** (k = 3): bài nặng hơn, cần DP/đếm thêm.
4. **Chỉ được dùng ván nguyên**: đáp án = `max(cnt.values())`.
5. **Cho tối thiểu độ cao `H`**: lọc `h >= H` trong `t`.

Ta cài thử biến thể 1 và 2 rồi kiểm tra với brute force.

In [24]:
def best_height(planks, allow_same_pair=True):
    cnt = Counter(planks)
    t = defaultdict(int)
    for x, v1 in cnt.items():
        t[x] += v1
        if allow_same_pair:
            t[2 * x] += v1 // 2
        for y, v2 in cnt.items():
            if y > x:
                t[x + y] += min(v1, v2)
    w = max(t.values())
    return w, min(h for h, v in t.items() if v == w)

print(best_height([1, 3, 2, 5, 7, 5, 4, 2, 1]))        # (4, 5)
print(best_height([2, 2, 2, 2], allow_same_pair=True))    # (4, 2): bốn tấm cao 2 tốt hơn hai tấm cao 4
print(best_height([2, 2, 3, 3], allow_same_pair=False))    # dùng nguyên hoặc ghép 2+3

# Kiểm tra biến thể 1 với brute force: giá trị rộng phải khớp, chiều cao phải đạt rộng đó
rng = random.Random(9)
for _ in range(300):
    planks = [rng.randint(1, 8) for _ in range(rng.randint(1, 8))]
    w, h = best_height(planks)
    assert w == brute_force(planks)
    cnt = Counter(planks)
    assert width_for_target(cnt, h) == w
print("Biến thể: OK")

(4, 5)
(4, 2)
(2, 2)
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force bitmask | thử mọi cách ghép theo định nghĩa | `O(n²·2^n·n)` | `O(2^n)` | trọng tài, `n ≤ 10` |
| 2. Duyệt từng `h` | với mỗi `h`, cộng ba nguồn | `O(m³)` | `O(m²)` | `m` nhỏ; dễ hiểu |
| 3. Cộng dồn (nộp) | mỗi cặp đẩy `min` vào `t[x+y]` | `O(n + m²)` | `O(m²)` | thi đấu |
| 4. Cộng dồn trên danh sách sắp | như 3 nhưng thứ tự xác định | `O(n log n + m²)` | `O(m²)` | cần trả cả chiều cao tối ưu |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `Counter([2, 2, 3])[2]` và `Counter([2, 2, 3])[9]` bằng bao nhiêu?

**Bài 2 (tính tay).** `planks = [1, 1, 2, 2, 3, 4]`. Với `h = 4`, có bao nhiêu tấm? Đáp án cuối cùng của bài là bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thiện `contribution(cnt, h)` trả về độ rộng tối đa với chiều cao mục tiêu `h`.

**Bài 4 (tự viết lại).** Viết `my_solve(planks)` bằng cách cộng dồn `t` (không nhìn code nộp).

**Bài 5 (biến thể / mini challenge).** Cho thêm số nguyên `K`. Viết `widest_at_least(planks, K)` trả về độ rộng lớn nhất trong các hàng rào có chiều cao `h >= K` (hoặc `0` nếu không có).

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def contribution(cnt, h):
    total = 0
    # TODO 1: cộng cnt.get(h, 0)
    # TODO 2: nếu h chẵn, cộng cnt.get(h // 2, 0) // 2
    # TODO 3: với mỗi x trong cnt, y = h - x; nếu x < y và y in cnt, cộng min(cnt[x], cnt[y])
    return total

# Bài 4 — TODO
def my_solve(planks):
    # TODO: Counter, defaultdict, hai vòng lồng, trả max(t.values())
    pass

# Bài 5 — TODO
def widest_at_least(planks, K):
    # TODO: như bài 4 nhưng chỉ lấy h >= K
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `Counter` đếm số lần.
- Mức 2: giá trị không có thì kết quả là `0`.
- Mức 3: `2` và `0`.

**Bài 2.**
- Mức 1: `h = 4` → nguyên `4` (1 ván), ghép `2+2` (2 ván cao 2 → 1 tấm), ghép `1+3` (`min(2,1)`).
- Mức 2: tổng `1 + 1 + 1 = 3`.
- Mức 3: thử `h = 3`: nguyên 1, `1+2` → `min(2,2) = 2` → tổng 3; `h = 2`: nguyên 2, `1+1` → 1 → tổng 3. Đáp án cuối `3`.

**Bài 3.**
- Mức 1: ba khối cộng độc lập.
- Mức 2: dùng `cnt.get(k, 0)`.
- Mức 3: vòng `for x in cnt: y = h - x; if x < y and y in cnt: total += min(cnt[x], cnt[y])`.

**Bài 4.**
- Mức 1: `t = defaultdict(int)`.
- Mức 2: `t[x] += v1; t[2*x] += v1 // 2`.
- Mức 3: vòng trong `if y > x: t[x + y] += min(v1, v2)`.

**Bài 5.**
- Mức 1: chỉ đổi dòng cuối.
- Mức 2: `max(v for h, v in t.items() if h >= K)`.
- Mức 3: dùng `default=0` trong `max`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Counter([2, 2, 3])[2] == 2 and Counter([2, 2, 3])[9] == 0

# Bài 2
assert width_for_target(Counter([1, 1, 2, 2, 3, 4]), 4) == 3
assert Solution().maximumWidth([1, 1, 2, 2, 3, 4]) == 3

# Bài 3
def contribution_ans(cnt, h):
    total = cnt.get(h, 0)
    if h % 2 == 0:
        total += cnt.get(h // 2, 0) // 2
    for x in cnt:
        y = h - x
        if x < y and y in cnt:
            total += min(cnt[x], cnt[y])
    return total

rng = random.Random(31)
for _ in range(200):
    planks = [rng.randint(1, 9) for _ in range(rng.randint(1, 12))]
    cnt = Counter(planks)
    for h in range(1, 20):
        assert contribution_ans(cnt, h) == width_for_target(cnt, h)

# Bài 4
def my_solve_ans(planks):
    cnt = Counter(planks)
    t = defaultdict(int)
    for x, v1 in cnt.items():
        t[x] += v1
        t[2 * x] += v1 // 2
        for y, v2 in cnt.items():
            if y > x:
                t[x + y] += min(v1, v2)
    return max(t.values())

for _ in range(300):
    planks = [rng.randint(1, 9) for _ in range(rng.randint(1, 9))]
    assert my_solve_ans(planks) == Solution().maximumWidth(planks) == brute_force(planks)
print("Bài 1–4 OK")

Bài 1–4 OK


In [27]:
# Bài 5
def widest_at_least_ans(planks, K):
    cnt = Counter(planks)
    t = defaultdict(int)
    for x, v1 in cnt.items():
        t[x] += v1
        t[2 * x] += v1 // 2
        for y, v2 in cnt.items():
            if y > x:
                t[x + y] += min(v1, v2)
    return max((v for h, v in t.items() if h >= K), default=0)

def widest_at_least_brute(planks, K):
    n = len(planks)
    best = 0
    cands = set(planks) | {planks[i] + planks[j] for i in range(n) for j in range(i + 1, n)}
    for h in cands:
        if h < K:
            continue
        @lru_cache(None)
        def go(mask):
            if mask == (1 << n) - 1:
                return 0
            i = 0
            while mask >> i & 1:
                i += 1
            res = go(mask | 1 << i)
            if planks[i] == h:
                res = max(res, 1 + go(mask | 1 << i))
            for j in range(i + 1, n):
                if not mask >> j & 1 and planks[i] + planks[j] == h:
                    res = max(res, 1 + go(mask | 1 << i | 1 << j))
            return res
        best = max(best, go(0))
    return best

assert widest_at_least_ans([1, 3, 2, 5, 7, 5, 4, 2, 1], 6) == 4   # h=7: 7 + (2+5)x2 + (3+4)
assert widest_at_least_ans([1, 3, 2, 5, 7, 5, 4, 2, 1], 100) == 0
rng = random.Random(55)
for _ in range(300):
    planks = [rng.randint(1, 8) for _ in range(rng.randint(1, 8))]
    k = rng.randint(1, 12)
    assert widest_at_least_ans(planks, k) == widest_at_least_brute(planks, k), (planks, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đếm rồi cộng dồn theo cặp":**

```python
from collections import Counter, defaultdict

cnt = Counter(arr)
t = defaultdict(int)
for x, v1 in cnt.items():
    t[x] += v1                       # dùng nguyên
    t[2 * x] += v1 // 2              # ghép hai phần tử cùng giá trị
    for y, v2 in cnt.items():
        if y > x:                    # mỗi cặp khác giá trị chỉ một lần
            t[x + y] += min(v1, v2)
return max(t.values())
```

**Checklist trước khi submit:**

- [ ] Đã đếm tần suất bằng `Counter`.
- [ ] Ba nguồn: nguyên, cùng cao `v // 2`, khác cao `min`.
- [ ] Vòng trong chỉ `y > x` để không đếm đôi.
- [ ] Đáp án là `max` theo `h`, không cộng mọi `h`.
- [ ] Đáp án luôn `≥ 1` vì có thể dùng một ván nguyên.
- [ ] Đã đối chiếu với brute force bitmask trên input nhỏ.